## Import Libraries

In [1]:
from pathlib import Path
import sys
import importlib
import itertools
import warnings
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

try:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Dense, Dropout, Input
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.optimizers import Adam

    tf.keras.utils.set_random_seed(RANDOM_STATE)
    print("TensorFlow version:", tf.__version__)
except ImportError as exc:
    raise ImportError(
        "TensorFlow is required for the Multilayer ANN. "
        "Install it with `pip install tensorflow`, restart the kernel, "
        "and run the notebook again."
    ) from exc

TensorFlow version: 2.21.0


## Locate the project and reuse the updated preprocessing pipeline

In [2]:
current_dir = Path.cwd().resolve()

candidate_roots = [
    current_dir,
    current_dir.parent,
    current_dir.parent.parent,
]

project_root = next(
    (
        path
        for path in candidate_roots
        if (
            path / "data" / "raw" /
            "GlobalLandTemperaturesByCountry.csv"
        ).exists()
        and (path / "code" / "preprocess.py").exists()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Cannot find the project root. Expected both:\n"
        "  data/raw/GlobalLandTemperaturesByCountry.csv\n"
        "  code/preprocess.py\n"
        "Place this notebook inside the same project used by "
        "01_data_preprocessing.ipynb."
    )

sys.path.insert(0, str(project_root / "code"))

import preprocess as pp
importlib.reload(pp)

print("Project root:", project_root)

Project root: C:\Users\dlihi\Documents\PRT565 Machine Learning\Assessment 3\PRT565-Machine-Learning


## Run the updated proprocessing pipeline

In [3]:
results = pp.run_pipeline(project_root)

train_df = results["train"].copy()
test_df = results["test"].copy()

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

print("\nTraining reference years:")
print(sorted(train_df["reference_year"].unique()))

print("\nTesting reference years:")
print(sorted(test_df["reference_year"].unique()))

print("\nTraining class distribution:")
display(
    train_df["target_exposure_class"]
    .value_counts()
    .reindex(["Low", "Medium", "High"])
)

print("\nTesting class distribution:")
display(
    test_df["target_exposure_class"]
    .value_counts()
    .reindex(["Low", "Medium", "High"])
)

display(train_df.head())
display(test_df.head())

Preprocessing completed successfully.
Model-ready files saved to: C:\Users\dlihi\Documents\PRT565 Machine Learning\Assessment 3\PRT565-Machine-Learning\data\processed
Training shape: (663, 13)
Testing shape: (221, 13)

Training reference years:
[np.int64(1970), np.int64(1980), np.int64(1990)]

Testing reference years:
[np.int64(2000)]

Training class distribution:


target_exposure_class
Low       221
Medium    221
High      221
Name: count, dtype: int64


Testing class distribution:


target_exposure_class
Low       62
Medium    93
High      66
Name: count, dtype: int64

,country,cca3,reference_year,population_at_reference,density_at_reference_per_km2,population_growth_prior_decade_pct,prior_decade_mean_temp_c,prior_decade_warming_c_per_decade,prior_decade_detrended_volatility_c,prior_decade_mean_temp_uncertainty_c,area_km2,continent,target_exposure_class
0,Aruba,ABW,1970,59106,328.366667,NaN,28.199408,0.107460,0.189841,0.358767,180,North America,High
1,Aruba,ABW,1980,62267,345.927778,5.348019,28.167542,0.197955,0.287829,0.327217,180,North America,Low
2,Aruba,ABW,1990,65712,365.066667,5.532626,28.342575,-0.270318,0.302661,0.324258,180,North America,High
4,Afghanistan,AFG,1970,10752971,16.486471,NaN,13.961283,-0.322576,0.494226,0.411983,652230,Asia,Low
5,Afghanistan,AFG,1980,12486631,19.144521,16.122614,14.035983,0.108848,0.746613,0.412525,652230,Asia,Low


,country,cca3,reference_year,population_at_reference,density_at_reference_per_km2,population_growth_prior_decade_pct,prior_decade_mean_temp_c,prior_decade_warming_c_per_decade,prior_decade_detrended_volatility_c,prior_decade_mean_temp_uncertainty_c,area_km2,continent,target_exposure_class
3,Aruba,ABW,2000,89101,495.005556,35.593195,28.531633,0.322152,0.224315,0.317167,180,North America,High
7,Afghanistan,AFG,2000,19542982,29.963329,82.733565,14.732458,0.799712,0.324100,0.451775,652230,Asia,Low
11,Angola,AGO,2000,16394062,13.149966,38.596362,22.477567,0.413253,0.294703,0.494600,1246700,Africa,Low
15,Anguilla,AIA,2000,11047,121.395604,32.840308,27.266525,0.514823,0.175331,0.282558,91,North America,High
19,Albania,ALB,2000,3182021,110.686691,-3.430736,13.149892,0.307763,0.498491,0.328900,28748,Europe,Medium
